# Physics-Informed Machine Learning
## Notebook 04 — Our First Physics-Informed Neural Network

### Problem

We want to solve:

    dy/dt = 2y

with the initial condition:

    y(0) = 1

We already know the analytical solution:

    y(t) = e^(2t)

But we will NOT give this solution to the neural network during training.

Instead, the neural network will learn a function:

    y_hat(t)

and we will require that:

    dy_hat/dt - 2y_hat = 0

while also requiring:

    y_hat(0) = 1

This is a Physics-Informed Neural Network (PINN).

# 1. The Main Idea

A normal neural network learns from labeled examples.

For example:

    t → y_true

and minimizes:

    Data Loss = MSE(y_hat, y_true)

A PINN can work differently.

We know the governing differential equation:

    dy/dt = 2y

So if the network predicts:

    y_hat(t)

we can calculate:

    dy_hat/dt

using automatic differentiation.

Then define the differential equation residual:

    R(t) = dy_hat/dt - 2y_hat

For a correct solution:

    R(t) = 0

Therefore the physics loss is:

    L_physics = mean(R(t)^2)

We also need the initial condition:

    y_hat(0) = 1

so:

    L_initial = (y_hat(0) - 1)^2

Finally:

    L_total = L_physics + L_initial

The neural network is trained to minimize this loss.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from scipy.integrate import solve_ivp

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch:", torch.__version__)

PyTorch: 2.13.0+cpu


# 2. Neural Network

The network receives:

    t

and outputs:

    y_hat(t)

Architecture:

    t
    ↓
    Linear
    ↓
    Tanh
    ↓
    Linear
    ↓
    Tanh
    ↓
    Linear
    ↓
    y_hat

In [2]:
class PINN(nn.Module):
    """
    Neural network representing the unknown function y(t).

    Input:
        t

    Output:
        y_hat(t)
    """

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1, 32),
            nn.Tanh(),

            nn.Linear(32, 32),
            nn.Tanh(),

            nn.Linear(32, 1)
        )

    def forward(self, t):
        return self.network(t)


model = PINN()

print(model)

PINN(
  (network): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): Tanh()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): Tanh()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


# 3. Collocation Points

Here is an important PINN concept.

We do not necessarily need a dataset containing:

    t → y

Instead, we choose points in the domain where we want the differential
equation to be satisfied.

These are called:

    collocation points

For example:

    t = 0.0
    t = 0.1
    t = 0.2
    ...
    t = 2.0

At every point, we ask:

    Does the neural network satisfy the ODE here?

We therefore need:

    t.requires_grad_(True)

because PyTorch must calculate:

    dy_hat/dt

In [3]:
# Domain of the problem
t_min = 0.0
t_max = 2.0

# Number of collocation points
N = 1000

# Create collocation points
t_collocation = torch.linspace(
    t_min,
    t_max,
    N
).reshape(-1, 1)

# We need derivatives with respect to t
t_collocation.requires_grad_(True)

print("Shape:", t_collocation.shape)
print("First points:")
print(t_collocation[:5])

Shape: torch.Size([1000, 1])
First points:
tensor([[0.0000],
        [0.0020],
        [0.0040],
        [0.0060],
        [0.0080]], grad_fn=<SliceBackward0>)
